In [1]:

import joblib
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:

RANDOM_STATE = 42

# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------
titanic = sns.load_dataset("titanic")

# ---------------------------------------------------------------------------
# Feature engineering
# (seaborn's titanic has no 'name' column, so we approximate "title" from
# the 'who' column instead of parsing "Mr./Mrs./Miss." out of a name string)
# ---------------------------------------------------------------------------
titanic["family_size"] = titanic["sibsp"] + titanic["parch"] + 1

who_to_title = {"man": "Mr", "woman": "Mrs", "child": "Child"}
titanic["title"] = titanic["who"].map(who_to_title).fillna("Other")

features = ["pclass", "sex", "age", "fare", "embarked", "family_size", "title"]
target = "survived"

X = titanic[features]
y = titanic[target]

# ---------------------------------------------------------------------------
# Train/test split — done BEFORE any imputing/encoding, this is the fix for
# the leakage in the original version (which imputed on the full dataframe)
# ---------------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# ---------------------------------------------------------------------------
# Preprocessing — imputers/encoders are fit only on X_train, inside the
# pipeline below, never on the full dataset
# ---------------------------------------------------------------------------
numeric_features = ["age", "fare", "family_size"]
categorical_features = ["pclass", "sex", "embarked", "title"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

# ---------------------------------------------------------------------------
# Model + pipeline
# Bundling preprocessing + model into one Pipeline means Streamlit can load
# ONE object and call .predict() directly on raw user input, no manual
# re-implementation of imputing/encoding at inference time.
# ---------------------------------------------------------------------------
pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE)),
])


In [3]:

# ---------------------------------------------------------------------------
# Hyperparameter tuning
# ---------------------------------------------------------------------------
param_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [4, 6, 8, None],
    "model__min_samples_leaf": [1, 2, 4],
}

grid = GridSearchCV(pipeline, param_grid, cv=5, scoring="roc_auc", n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print("Best params:", grid.best_params_)
print("Best CV ROC-AUC:", round(grid.best_score_, 4))

# ---------------------------------------------------------------------------
# Evaluate
# ---------------------------------------------------------------------------
y_pred_train = best_model.predict(X_train)
y_pred_test = best_model.predict(X_test)
y_proba_test = best_model.predict_proba(X_test)[:, 1]

print("\nTraining accuracy:", round(accuracy_score(y_train, y_pred_train) * 100, 2), "%")
print("Testing accuracy: ", round(accuracy_score(y_test, y_pred_test) * 100, 2), "%")
print("Testing ROC-AUC:  ", round(roc_auc_score(y_test, y_proba_test), 4))
print("\nClassification report (test):\n",
      classification_report(y_test, y_pred_test, target_names=["Died", "Survived"]))


Best params: {'model__max_depth': None, 'model__min_samples_leaf': 4, 'model__n_estimators': 400}
Best CV ROC-AUC: 0.8757

Training accuracy: 87.64 %
Testing accuracy:  81.56 %
Testing ROC-AUC:   0.8533

Classification report (test):
               precision    recall  f1-score   support

        Died       0.80      0.93      0.86       110
    Survived       0.85      0.64      0.73        69

    accuracy                           0.82       179
   macro avg       0.82      0.78      0.79       179
weighted avg       0.82      0.82      0.81       179



In [4]:
# ---------------------------------------------------------------------------
# Save the fitted pipeline for Streamlit to load
# ---------------------------------------------------------------------------
joblib.dump(best_model, "titanic_pipeline.joblib")
print("Saved pipeline to titanic_pipeline.joblib")


Saved pipeline to titanic_pipeline.joblib
